# NB-15: WanS2VDiTBlock and AudioInjector -- S2V Block Variant

## Learning Objectives

1. **Understand how WanS2VDiTBlock inherits DiTBlock**, overriding only `forward()` to add dual-timestep per-position modulation -- no new parameters, no new sub-modules.
2. **Trace the AdaLayerNorm shift/scale conditioning** from the global audio embedding, distinguishing it from adaLN-Zero (which conditions on timestep).
3. **See how AudioInjector performs per-frame cross-attention** by reshaping content tokens from `[B, F*H*W, D]` to `[(B*F), H*W, D]`, ensuring each frame attends to its temporally aligned audio features.

## Prerequisites

- **NB-06** (DiTBlock forward pass) -- self-attention, cross-attention, FFN with adaLN-Zero modulation
- **NB-04** (CrossAttention) -- query/key/value projections, multi-head attention
- **NB-13** (Audio encoding) -- CausalAudioEncoder output shapes: global `[B, F, 1, dim]`, local `[B, F, num_audio_token+1, dim]`

## Concept Map

```
WanS2VDiTBlock overrides DiTBlock.forward()
  to split t_mod by position:
    content tokens -> real timestep
    ref+motion tokens -> zero timestep
         |
         v
AudioInjector_WAN applies per-frame audio cross-attention
  after specified blocks (audio_inject_layers=[0, 2]):
    AdaLayerNorm conditioning from global audio
    CrossAttention with local audio features
         |
         v
NB-16 composes the full WanS2VModel forward pass
```

In [ ]:
# Source: wan_video_dit_s2v.py -- S2V module loading setup
import sys, types, importlib.util, pathlib
import torch
import torch.nn as nn
import torch.nn.functional as F

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for S2V block demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py (base DiT module, dependency of S2V)
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit_mod
_spec.loader.exec_module(dit_mod)

# Load wan_video_dit_s2v.py (S2V model with block and audio injector classes)
_s2v_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit_s2v.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit_s2v", _s2v_path)
s2v_mod = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit_s2v"] = s2v_mod
_spec.loader.exec_module(s2v_mod)

from diffsynth.models.wan_video_dit_s2v import WanS2VDiTBlock, AudioInjector_WAN, AdaLayerNorm
from diffsynth.models.wan_video_dit import DiTBlock, CrossAttention, modulate, sinusoidal_embedding_1d, precompute_freqs_cis_3d
from diffsynth.models.wan_video_dit_s2v import torch_dfs

print(f"Imported: WanS2VDiTBlock, AudioInjector_WAN, AdaLayerNorm")
print(f"Imported: DiTBlock, CrossAttention, modulate, sinusoidal_embedding_1d")
print(f"Imported: torch_dfs")
print("Setup complete.")

## S2V Block Loop -- Dual Timestep + Audio Injection

```
S2V Block Loop -- Dual Timestep + Audio Injection
===================================================

For each block_id in range(num_layers):

  STEP 1: WanS2VDiTBlock.forward(x, context, t_mod, seq_len_x, freqs)
  -----------------------------------------------------------------------
  t_mod [1, 6, 2, dim]     -- index 0 = real timestep, index 1 = zero timestep
           |
           +-- add learned modulation, then chunk into 6 modulation parameters
           |
           +-- Per-position split:
                 tokens 0..seq_len_x-1     (content) -> t_mod[:,:,0] (REAL timestep)
                 tokens seq_len_x..end     (ref+motion) -> t_mod[:,:,1] (ZERO timestep)
           |
           +-- modulate -> self_attn -> cross_attn -> FFN  (same as DiTBlock)

  STEP 2: after_transformer_block(block_id, x, audio_global, audio_local, seq_len_x)
  -----------------------------------------------------------------------
  IF block_id in audio_inject_layers:
    content_tokens = x[:, :seq_len_x]                      [B, F*H*W, dim]
    reshape per-frame:  rearrange("b (t n) c -> (b t) n c") [(B*F), H*W, dim]
    AdaLayerNorm(tokens, audio_global[:, 0])                [(B*F), H*W, dim]
    CrossAttention(tokens, audio_local)                     [(B*F), H*W, dim]
    reshape back: rearrange("(b t) n c -> b (t n) c")       [B, F*H*W, dim]
    x[:, :seq_len_x] += residual
```

Source: `diffsynth/models/wan_video_dit_s2v.py`, lines 341-356, 459-482

In [ ]:
# Reduced config (per D-09, consistent with all prior phases)
dim = 384
num_heads = 4
head_dim = dim // num_heads  # 96
ffn_dim = 1024
freq_dim = 256

# S2V block config (per D-07)
num_layers = 3
audio_inject_layers = [0, 2]  # inject at first and last of 3 blocks

# Sequence dimensions
F_content = 4    # content frames (after patchify)
H_patch = 8      # spatial height after patchify (16 / patch_size[1]=2)
W_patch = 8      # spatial width after patchify
S_content = F_content * H_patch * W_patch  # 256
S_ref = 64       # reference image tokens (1 frame * 8 * 8)
S_motion = 84    # motion tokens from FramePackMotioner (NB-14)
S_total = S_content + S_ref + S_motion  # 404

# Audio config (per D-08, from NB-13)
num_audio_token = 2
audio_dim = 384

print(f"S_content={S_content}, S_ref={S_ref}, S_motion={S_motion}")
print(f"S_total={S_total}  (content + ref + motion)")
print(f"audio_inject_layers={audio_inject_layers}")

## Brief DiTBlock Recap

DiTBlock (see NB-06) wires 5 primitives: RMSNorm -> self-attention -> cross-attention -> FFN,
with adaLN-Zero modulation and gated residuals.

```python
def forward(self, x, context, t_mod, freqs) -> x  # [B, S, dim]
# t_mod: [B, 6, dim] -- single timestep modulation (6 params: shift/scale/gate for attn + FFN)
```

WanS2VDiTBlock inherits DiTBlock and overrides ONLY the `forward()` method. It adds:
1. **Dual timestep:** `t_mod` has shape `[1, 6, 2, dim]` instead of `[B, 6, dim]`
2. **Per-position splitting:** first `seq_len_x` tokens get real timestep, rest get zero timestep

All sub-modules (`self_attn`, `cross_attn`, `ffn`, norms, `gate`, `modulation`) are inherited unchanged.

Source: `diffsynth/models/wan_video_dit.py`, line 197

## Inheritance Visualization

```
DiTBlock (NB-06)
  |
  +-- VaceWanAttentionBlock (NB-11)
  |     Adds: before_proj, after_proj
  |     Wraps: forward() with stack/unstack accumulation
  |
  +-- WanS2VDiTBlock (this notebook)
        Adds: NOTHING (no new sub-modules!)
        Overrides: forward() with dual-timestep per-position split
        Inherits: ALL sub-modules unchanged
```

In [ ]:
# Source: wan_video_dit_s2v.py, line 341
block = WanS2VDiTBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)
base_block = DiTBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)

# WanS2VDiTBlock has EXACTLY the same sub-modules as DiTBlock
s2v_modules = set(name for name, _ in block.named_modules())
base_modules = set(name for name, _ in base_block.named_modules())
assert s2v_modules == base_modules, f"Module sets differ: {s2v_modules.symmetric_difference(base_modules)}"
print(f"WanS2VDiTBlock modules == DiTBlock modules: {s2v_modules == base_modules}")
print(f"Total sub-modules: {len(s2v_modules)}")

# Same parameter count
p_s2v = sum(p.numel() for p in block.parameters())
p_base = sum(p.numel() for p in base_block.parameters())
assert p_s2v == p_base, f"Param count differs: S2V={p_s2v}, base={p_base}"
print(f"Parameters: S2V={p_s2v:,} == base={p_base:,}  (no new parameters!)")

# Verify inheritance
assert isinstance(block, DiTBlock), "WanS2VDiTBlock must be a DiTBlock subclass"
print(f"isinstance(WanS2VDiTBlock, DiTBlock): True")

## 1. AdaLayerNorm -- Audio-Conditioned Normalization

Before tracing AudioInjector, we need to understand AdaLayerNorm. It is distinct from
adaLN-Zero (NB-05):

| Property | adaLN-Zero (DiTBlock) | AdaLayerNorm (AudioInjector) |
|----------|----------------------|------------------------------|
| Conditioning source | Timestep (`nn.Parameter` + `time_projection`) | Audio embedding (linear projection) |
| Parameters | 6 params: shift/scale/gate x2 | 2 params: shift/scale |
| Activation | None (direct `nn.Parameter`) | SiLU-gated linear |
| Usage | Every DiTBlock, every forward | Only at audio injection layers |

Source: `diffsynth/models/wan_video_dit_s2v.py`, line 259

In [ ]:
# Source: wan_video_dit_s2v.py, line 259
adaln = AdaLayerNorm(embedding_dim=dim, output_dim=dim * 2)
print(f"AdaLayerNorm: linear({dim} -> {dim * 2}), norm({dim})")
print(f"  output_dim = dim * 2 because it produces shift AND scale")

# Trace forward
B = 1
x_adaln = torch.randn(B, S_content // F_content, dim)  # [1, 64, 384] (one frame spatial tokens)
temb = torch.randn(B, dim)  # global audio embedding for one frame

adaln.eval()
with torch.no_grad():
    # Step 1: SiLU + linear projection
    temb_proj = adaln.linear(F.silu(temb))
    print(f"\nStep 1 -- linear(SiLU(temb)):  {temb.shape} -> {temb_proj.shape}")
    assert temb_proj.shape == torch.Size([B, dim * 2])  # [1, 768]

    # Step 2: chunk into shift and scale
    shift, scale = temb_proj.chunk(2, dim=1)
    print(f"Step 2 -- chunk(2):            shift={shift.shape}, scale={scale.shape}")
    assert shift.shape == torch.Size([B, dim])
    assert scale.shape == torch.Size([B, dim])

    # Step 3: apply to normalized input
    x_normed = adaln.norm(x_adaln)
    x_out = x_normed * (1 + scale[:, None, :]) + shift[:, None, :]
    print(f"Step 3 -- norm(x) * (1+scale) + shift: {x_out.shape}")
    assert x_out.shape == x_adaln.shape

    # Verify matches direct call
    x_direct = adaln(x_adaln, temb)
    assert torch.allclose(x_direct, x_out, atol=1e-6), "Manual trace must match direct call"
    print(f"\nVerification: manual trace matches direct call")

## 2. WanS2VDiTBlock -- Dual-Timestep Per-Position Splitting

The key innovation in WanS2VDiTBlock is the dual-timestep mechanism. In the base DiTBlock,
ALL tokens receive the same timestep modulation. In S2V:
- **Content tokens** get the REAL diffusion timestep (they are being denoised)
- **Reference + motion tokens** get timestep ZERO (they are clean conditioning signals)

This makes architectural sense: only content tokens carry noise that needs to be predicted.
Reference and motion tokens are already clean -- they should not be modulated by the noise level.

Source: `diffsynth/models/wan_video_dit_s2v.py`, line 343

In [ ]:
# Source: wan_video_dit_s2v.py, lines 544-546 (in WanS2VModel.forward)
# Trace how t_mod [1, 6, 2, dim] is constructed

timestep = torch.tensor([0.5])  # real diffusion timestep

# Step 1: cat real timestep with zero
timestep_dual = torch.cat([timestep, torch.zeros([1])])
print(f"Step 1 -- cat([0.5, 0.0]):             {timestep_dual.shape}  values={timestep_dual.tolist()}")
assert timestep_dual.shape == torch.Size([2])

# Step 2: sinusoidal embedding
t_emb = sinusoidal_embedding_1d(freq_dim, timestep_dual)
print(f"Step 2 -- sinusoidal_embedding({freq_dim}): {t_emb.shape}")
assert t_emb.shape == torch.Size([2, freq_dim])  # [2, 256]

# Step 3: time_embedding MLP
time_embedding = nn.Sequential(nn.Linear(freq_dim, dim), nn.SiLU(), nn.Linear(dim, dim))
with torch.no_grad():
    t = time_embedding(t_emb)
print(f"Step 3 -- time_embedding:              {t.shape}")
assert t.shape == torch.Size([2, dim])  # [2, 384]

# Step 4: time_projection
time_projection = nn.Sequential(nn.SiLU(), nn.Linear(dim, dim * 6))
with torch.no_grad():
    t_proj = time_projection(t)
print(f"Step 4 -- time_projection:             {t_proj.shape}")
assert t_proj.shape == torch.Size([2, dim * 6])  # [2, 2304]

# Step 5: unflatten to 6 modulation parameters
t_mod_pre = t_proj.unflatten(1, (6, dim))
print(f"Step 5 -- unflatten(1, (6, {dim})):    {t_mod_pre.shape}")
assert t_mod_pre.shape == torch.Size([2, 6, dim])  # [2, 6, 384]

# Step 6: unsqueeze + transpose to get [1, 6, 2, dim]
# unsqueeze(2): [2, 6, 384] -> [2, 6, 1, 384]
# transpose(0, 2): swap dim-0 (batch=2) with dim-2 (1) -> [1, 6, 2, 384]
t_mod = t_mod_pre.unsqueeze(2).transpose(0, 2)
print(f"Step 6 -- unsqueeze(2).transpose(0,2): {t_mod.shape}")
assert t_mod.shape == torch.Size([1, 6, 2, dim])

print(f"\nt_mod[:,:,0,:] = REAL timestep modulation (for content)")
print(f"t_mod[:,:,1,:] = ZERO timestep modulation (for ref+motion)")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 343-350
# Trace how t_mod is split per position inside WanS2VDiTBlock.forward

seq_len_x = S_content  # 256 content tokens
x_total = torch.randn(1, S_total, dim)  # [1, 404, 384]

block.eval()
with torch.no_grad():
    # Source: wan_video_dit_s2v.py, line 344
    # Add learned modulation (nn.Parameter [1,6,dim]) to t_mod [1,6,2,dim], then chunk
    # modulation.unsqueeze(2) -> [1, 6, 1, dim] broadcasts to [1, 6, 2, dim]
    t_mod_with_learned = (block.modulation.unsqueeze(2).to(dtype=t_mod.dtype) + t_mod).chunk(6, dim=1)
    print(f"After adding learned modulation and chunking: {len(t_mod_with_learned)} elements")
    print(f"Each element shape: {t_mod_with_learned[0].shape}")
    assert t_mod_with_learned[0].shape == torch.Size([1, 1, 2, dim])

    # Per-position split: content gets index 0, ref+motion gets index 1
    # element[:, :, 0] selects real timestep -> [1, 1, dim] -> expand to [1, seq_len_x, dim]
    # element[:, :, 1] selects zero timestep -> [1, 1, dim] -> expand to [1, S_total-seq_len_x, dim]
    t_mod_split = [
        torch.cat([
            element[:, :, 0].expand(1, seq_len_x, dim),
            element[:, :, 1].expand(1, S_total - seq_len_x, dim)
        ], dim=1)
        for element in t_mod_with_learned
    ]

    print(f"\nAfter per-position split:")
    print(f"  Each element: {t_mod_split[0].shape}")
    assert t_mod_split[0].shape == torch.Size([1, S_total, dim])

    # Verify content tokens get real timestep, ref+motion get zero
    print(f"  Content tokens (0:{seq_len_x}): real timestep modulation")
    print(f"  Ref+motion tokens ({seq_len_x}:{S_total}): zero timestep modulation")
    print(f"  Total: {seq_len_x} + {S_total - seq_len_x} = {S_total}")

    # Verify the 6 parameters are named correctly
    shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = t_mod_split
    assert shift_msa.shape == torch.Size([1, S_total, dim])
    assert gate_mlp.shape == torch.Size([1, S_total, dim])
    print(f"\n6 modulation params: shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp")
    print(f"All have shape: [1, {S_total}, {dim}]  (per-token modulation)")

## 3. AudioInjector_WAN -- Per-Frame Cross-Attention

AudioInjector operates AFTER specified DiT blocks (not inside them). It:
1. Takes only the content tokens (first `seq_len_x` tokens)
2. Reshapes them per-frame: `[B, F*H*W, D]` -> `[(B*F), H*W, D]`
3. Applies AdaLayerNorm conditioning from global audio embedding
4. Runs CrossAttention where each frame's spatial tokens attend to that frame's audio tokens
5. Adds the result back as a residual

This per-frame structure ensures each video frame attends to its temporally aligned audio features.

Source: `diffsynth/models/wan_video_dit_s2v.py`, line 281

In [ ]:
# Source: wan_video_dit_s2v.py, lines 281-318
# AudioInjector uses torch_dfs to find DiTBlock instances in the module tree

# Create a small block list (3 blocks, matching num_layers=3)
blocks = nn.ModuleList([
    WanS2VDiTBlock(has_image_input=False, dim=dim, num_heads=num_heads, ffn_dim=ffn_dim)
    for _ in range(num_layers)
])

# Source: wan_video_dit_s2v.py, line 9
# torch_dfs walks the module tree to find blocks
all_modules, all_modules_names = torch_dfs(blocks, parent_name="root.transformer_blocks")
print(f"torch_dfs found {len(all_modules)} modules")

# Build AudioInjector
audio_injector = AudioInjector_WAN(
    all_modules=all_modules,
    all_modules_names=all_modules_names,
    dim=dim,
    num_heads=num_heads,
    inject_layer=audio_inject_layers,
    enable_adain=True,
    adain_dim=dim,
)
print(f"\nAudioInjector_WAN:")
print(f"  injected_block_id mapping: {audio_injector.injected_block_id}")
print(f"  Number of CrossAttention modules: {len(audio_injector.injector)}")
print(f"  Number of AdaLayerNorm modules: {len(audio_injector.injector_adain_layers)}")

assert audio_injector.injected_block_id == {0: 0, 2: 1}, f"Expected {{0: 0, 2: 1}}, got {audio_injector.injected_block_id}"
assert len(audio_injector.injector) == 2
assert len(audio_injector.injector_adain_layers) == 2
print(f"\nBlock 0 -> audio_injector[0], Block 2 -> audio_injector[1]")
print(f"Block 1 -> no audio injection")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 467-478
# Trace the per-frame reshape step by step

# Simulated inputs (from NB-13 output shapes)
hidden_all = torch.randn(1, S_total, dim)  # [1, 404, 384]
audio_emb_global = torch.randn(1, F_content, 1, dim)  # [1, 4, 1, 384]
audio_emb_local = torch.randn(1, F_content, num_audio_token + 1, dim)  # [1, 4, 3, 384]

from einops import rearrange

audio_injector.eval()
with torch.no_grad():
    # Step 1: Extract content tokens only
    content_tokens = hidden_all[:, :S_content].clone()
    print(f"Step 1 -- content tokens: {content_tokens.shape}  (first {S_content} of {S_total})")
    assert content_tokens.shape == torch.Size([1, S_content, dim])  # [1, 256, 384]

    # Step 2: Reshape per-frame (CRITICAL: S_content must be divisible by F_content)
    assert S_content % F_content == 0, f"{S_content} not divisible by {F_content}"
    tokens_per_frame = S_content // F_content
    content_per_frame = rearrange(content_tokens, "b (t n) c -> (b t) n c", t=F_content)
    print(f"Step 2 -- per-frame reshape: {content_tokens.shape} -> {content_per_frame.shape}")
    print(f"  {F_content} frames x {tokens_per_frame} spatial tokens/frame")
    assert content_per_frame.shape == torch.Size([F_content, tokens_per_frame, dim])  # [4, 64, 384]

    # Step 3: Reshape audio embeddings per-frame
    audio_global_pf = rearrange(audio_emb_global, "b t n c -> (b t) n c")
    audio_local_pf = rearrange(audio_emb_local, "b t n c -> (b t) n c", t=F_content)
    print(f"Step 3 -- audio per-frame:")
    print(f"  global: {audio_emb_global.shape} -> {audio_global_pf.shape}")
    print(f"  local:  {audio_emb_local.shape} -> {audio_local_pf.shape}")
    assert audio_global_pf.shape == torch.Size([F_content, 1, dim])  # [4, 1, 384]
    assert audio_local_pf.shape == torch.Size([F_content, num_audio_token + 1, dim])  # [4, 3, 384]

    # Step 4: AdaLayerNorm conditioning
    audio_attn_id = 0  # first injection point
    adain_out = audio_injector.injector_adain_layers[audio_attn_id](
        content_per_frame, temb=audio_global_pf[:, 0]
    )
    print(f"Step 4 -- AdaLayerNorm: {content_per_frame.shape} -> {adain_out.shape}")
    assert adain_out.shape == content_per_frame.shape

    # Step 5: CrossAttention (each frame attends to its audio)
    residual = audio_injector.injector[audio_attn_id](adain_out, audio_local_pf)
    print(f"Step 5 -- CrossAttention(Q={adain_out.shape}, K/V={audio_local_pf.shape}) -> {residual.shape}")
    assert residual.shape == torch.Size([F_content, tokens_per_frame, dim])

    # Step 6: Reshape back and add residual
    residual_full = rearrange(residual, "(b t) n c -> b (t n) c", t=F_content)
    print(f"Step 6 -- reshape back: {residual.shape} -> {residual_full.shape}")
    assert residual_full.shape == torch.Size([1, S_content, dim])

    hidden_all[:, :S_content] = hidden_all[:, :S_content] + residual_full
    print(f"\nResidual added to content tokens. Ref+motion tokens unchanged.")

In [ ]:
# Source: wan_video_dit_s2v.py, line 343
# Run a full WanS2VDiTBlock forward to verify shapes

# Source: wan_video_dit.py, line 75 (precompute_freqs_cis_3d)
# Build RoPE frequencies for the unified sequence
# For standalone testing, create enough freq positions for S_total tokens
f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(head_dim)

# Build 3D grid of freqs and reshape to [S, 1, head_dim//2]
# Use a grid large enough to cover S_total positions
F_grid, H_grid, W_grid = 8, 8, 8  # 8*8*8=512 > S_total=404
freqs_grid = torch.cat([
    f_freqs[:F_grid].view(F_grid, 1, 1, -1).expand(F_grid, H_grid, W_grid, -1),
    h_freqs[:H_grid].view(1, H_grid, 1, -1).expand(F_grid, H_grid, W_grid, -1),
    w_freqs[:W_grid].view(1, 1, W_grid, -1).expand(F_grid, H_grid, W_grid, -1),
], dim=-1).reshape(F_grid * H_grid * W_grid, 1, -1)
# Slice to S_total positions
freqs_simple = freqs_grid[:S_total]
print(f"RoPE freqs: {freqs_simple.shape}  (S_total={S_total}, head_dim//2={head_dim//2})")
assert freqs_simple.shape[0] == S_total

x_block = torch.randn(1, S_total, dim)
context_block = torch.randn(1, 10, dim)

block.eval()
with torch.no_grad():
    x_out = block(x_block, context_block, t_mod, seq_len_x=S_content, freqs=freqs_simple)

assert x_out.shape == torch.Size([1, S_total, dim]), f"Expected [1, {S_total}, {dim}], got {x_out.shape}"
print(f"WanS2VDiTBlock forward: {x_block.shape} -> {x_out.shape}")
print(f"  seq_len_x={S_content} (content tokens get real timestep)")
print(f"  remaining {S_total - S_content} tokens get zero timestep")

In [ ]:
# Source: wan_video_dit_s2v.py, lines 553-587
# Simulate the complete block loop as it runs in WanS2VModel.forward

x_loop = torch.randn(1, S_total, dim)
context_loop = torch.randn(1, 10, dim)

print(f"Block loop: {num_layers} blocks, audio injection at layers {audio_inject_layers}")
print(f"Input: {x_loop.shape}\n")

with torch.no_grad():
    for block_id, blk in enumerate(blocks):
        # Step 1: WanS2VDiTBlock forward
        x_loop = blk(x_loop, context_loop, t_mod, seq_len_x=S_content, freqs=freqs_simple)

        # Step 2: Audio injection (if this layer is in audio_inject_layers)
        if block_id in audio_injector.injected_block_id:
            audio_attn_id = audio_injector.injected_block_id[block_id]
            num_frames = audio_emb_local.shape[1]  # F_content = 4

            input_hidden = x_loop[:, :S_content].clone()
            input_hidden = rearrange(input_hidden, "b (t n) c -> (b t) n c", t=num_frames)

            audio_g = rearrange(audio_emb_global, "b t n c -> (b t) n c")
            adain_h = audio_injector.injector_adain_layers[audio_attn_id](input_hidden, temb=audio_g[:, 0])

            audio_l = rearrange(audio_emb_local, "b t n c -> (b t) n c", t=num_frames)
            res = audio_injector.injector[audio_attn_id](adain_h, audio_l)
            res = rearrange(res, "(b t) n c -> b (t n) c", t=num_frames)
            x_loop[:, :S_content] = x_loop[:, :S_content] + res

            print(f"  Block {block_id}: DiT forward + audio injection (injector[{audio_attn_id}])")
        else:
            print(f"  Block {block_id}: DiT forward only")

        assert x_loop.shape == torch.Size([1, S_total, dim])

print(f"\nOutput: {x_loop.shape}  (all {num_layers} blocks processed)")

## Summary

### Key Takeaways

1. **WanS2VDiTBlock inherits ALL DiTBlock sub-modules unchanged** -- it adds zero new parameters, only overrides `forward()` to implement dual-timestep per-position modulation.
2. **The dual-timestep mechanism** creates `t_mod [1, 6, 2, dim]` where index 0 is the real timestep (for content tokens being denoised) and index 1 is zero (for clean ref/motion tokens).
3. **AdaLayerNorm conditions hidden states via shift/scale from audio**, distinct from adaLN-Zero which conditions via timestep. It uses `SiLU(temb) -> Linear -> chunk(2)` to produce shift and scale.
4. **AudioInjector reshapes content tokens per-frame for CrossAttention**, requiring `S_content` to be divisible by `F_content`. Each frame attends independently to its temporally aligned audio features.

### Source References

| Symbol | Location |
|--------|----------|
| WanS2VDiTBlock | `wan_video_dit_s2v.py`, line 341 |
| WanS2VDiTBlock.forward | `wan_video_dit_s2v.py`, line 343 |
| AdaLayerNorm | `wan_video_dit_s2v.py`, line 259 |
| AudioInjector_WAN | `wan_video_dit_s2v.py`, line 281 |
| after_transformer_block | `wan_video_dit_s2v.py`, line 459 |
| torch_dfs | `wan_video_dit_s2v.py`, line 9 |
| DiTBlock (base class) | `wan_video_dit.py`, line 197 |

### Next

NB-16 shows the complete WanS2VModel forward pass, composing all S2V components:
audio encoding, motion packing, dual-timestep block loop with audio injection, and
output head.

## Exercises

### Exercise 1 -- What happens with audio_inject_layers=[0, 1, 2]?

What if you set `audio_inject_layers=[0, 1, 2]` (every block)? How many CrossAttention +
AdaLayerNorm module pairs would AudioInjector create? What is the computational cost
tradeoff vs. the default `[0, 2]`?

**Hint:** Each injection adds one CrossAttention and one AdaLayerNorm. With 3 inject layers,
you get 3 pairs instead of 2. The extra cross-attention per block increases compute but
gives the model more opportunities to incorporate audio information. The production model
uses 12 injection layers out of 40 blocks.

### Exercise 2 -- Why zero timestep for reference tokens?

In the dual-timestep mechanism, content tokens get the real timestep and ref/motion get zero.
What would happen if ALL tokens got the real timestep (no splitting)? Why is this
architecturally wrong for reference frames?

**Hint:** Reference frames are clean, not noisy. Applying noise-level-dependent modulation
to clean tokens would incorrectly scale their contribution based on the diffusion timestep.
At high noise levels (early denoising), the modulation would over-suppress the reference
signal that the model needs most.

### Exercise 3 -- Per-frame reshape divisibility

The per-frame reshape requires `S_content % F_content == 0`. If `F_content=3` and
spatial is 8x8, then `S_content = 3 * 64 = 192`. Verify this divides evenly.
What if `patch_size` changed to `(1, 4, 4)` making spatial `4x4`? Then
`S_content = 3 * 16 = 48`. Does the constraint still hold?

**Hint:** The constraint always holds because `S_content = F_content * H_patch * W_patch`
by construction. The divisibility is guaranteed by how patchification works.

## Coming Up

We have now covered all the building blocks of the S2V architecture:
- **NB-13:** Audio encoding (CausalAudioEncoder -> global + local embeddings)
- **NB-14:** Motion encoding (FramePackMotioner -> 84 multi-scale tokens + per-token RoPE)
- **This notebook:** Block modifications (dual-timestep splitting + per-frame audio injection)

In NB-16, we will compose everything into the complete WanS2VModel forward pass -- seeing how
the unified sequence of content + reference + motion tokens flows through the transformer
with audio injection, then gets sliced back to produce the output video.